# Phases 8-13: FPMC, Hybrid Fusion, Ablation & Final Evaluation (Extended)

This comprehensive notebook covers:
- **Phase 8**: Sequential Recommendation (FPMC)
- **Phase 9**: Full Hybrid Integration
- **Phase 10**: Hyperparameter Tuning
- **Phase 11**: Ablation Study (**Extended** — includes Pure FPMC and Pure KG w=1)
- **Phase 12**: Evaluation & Analysis
- **Phase 13**: Research Validation

**Run Time**: ~30 mins  
**Input**: All outputs from Phases 1-7  
**Output**: Final models, metrics, ablations, visualizations

In [ ]:
# Cell 1: Imports
import os
import pickle
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# GPU setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Environment detection - local vs Kaggle
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/kaggle-phase2-phase3-outputs/train.csv'):
    OUTPUT_DIR = '/kaggle/working'
else:
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-8-13-results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Cell 2: Load all models
print("Loading Phase 1-7 outputs...")

BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/datasets/chandrimanandi/phase-2-3-results'):
    phase23_dir = '/kaggle/input/datasets/chandrimanandi/phase-2-3-results'
    phase4_dir  = '/kaggle/input/datasets/chandrimanandi/phase-4-results'
    phase7_dir  = '/kaggle/input/datasets/chandrimanandi/phase-7-results'
else:
    phase23_dir = os.path.join(BASE_DIR, 'output')
    phase4_dir  = os.path.join(BASE_DIR, 'output', 'phase-4-results')
    phase7_dir  = os.path.join(BASE_DIR, 'output', 'phase-7-results')

train_df = pd.read_csv(os.path.join(phase23_dir, 'train.csv'))
val_df   = pd.read_csv(os.path.join(phase23_dir, 'val.csv'))
test_df  = pd.read_csv(os.path.join(phase23_dir, 'test.csv'))

with open(os.path.join(phase23_dir, 'id_maps.pkl'), 'rb') as f:
    id_maps = pickle.load(f)

with open(os.path.join(phase4_dir, 'biased_svd_model.pkl'), 'rb') as f:
    svd_model = pickle.load(f)

with open(os.path.join(phase7_dir, 'n2v_embeddings.pkl'), 'rb') as f:
    n2v_data = pickle.load(f)

n_users = id_maps['n_users']
n_items = id_maps['n_items']
P      = svd_model['P']
Q      = svd_model['Q']
bu     = svd_model['bu']
bi     = svd_model['bi']
r_mean = svd_model['global_mean']

n2v_item_emb_norm = n2v_data['item_embeddings_normalized']

print(f"✓ All models loaded")
print(f"  Users: {n_users}, Items: {n_items}")
print(f"  SVD dims: P={P.shape}, Q={Q.shape}")
print(f"  node2vec dims: {n2v_item_emb_norm.shape}")

## PHASE 8: Sequential Recommendation (FPMC)

In [ ]:
# Cell 3: FPMC Model
print("\n" + "="*60)
print("PHASE 8: FPMC - SEQUENTIAL RECOMMENDATION")
print("="*60)

class FPMC(nn.Module):
    """Factorizing Personalized Markov Chains"""
    def __init__(self, n_users, n_items, k_factors):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.k = k_factors
        self.M = nn.Parameter(torch.randn(n_users, k_factors) * 0.01)
        self.N = nn.Parameter(torch.randn(n_items, k_factors) * 0.01)

    def forward(self, user_idx, prev_item_idx, target_item_idx):
        m_u    = self.M[user_idx]
        n_prev = self.N[prev_item_idx]
        n_tgt  = self.N[target_item_idx]
        return ((m_u + n_prev) * n_tgt).sum(dim=1)

K_FPMC = P.shape[1]
fpmc_model = FPMC(n_users, n_items, K_FPMC).to(device)
print(f"FPMC model initialized with k={K_FPMC}")

In [ ]:
# Cell 4: Prepare FPMC training data
print("Preparing FPMC training data...")

train_sorted = train_df.sort_values(['user_idx', 'timestamp']).reset_index(drop=True)

fpmc_triples = []
for user_id in tqdm(range(n_users), desc="Creating FPMC triples"):
    user_seq = train_sorted[train_sorted['user_idx'] == user_id]
    if len(user_seq) >= 2:
        items = user_seq['item_idx'].values.astype(int)
        for i in range(len(items) - 1):
            fpmc_triples.append((int(user_id), int(items[i]), int(items[i+1])))

print(f"FPMC triples: {len(fpmc_triples)}")

if fpmc_triples:
    fpmc_triples = np.array(fpmc_triples)
    fpmc_users = torch.LongTensor(fpmc_triples[:, 0]).to(device)
    fpmc_prev  = torch.LongTensor(fpmc_triples[:, 1]).to(device)
    fpmc_next  = torch.LongTensor(fpmc_triples[:, 2]).to(device)

In [ ]:
# Cell 5: Train FPMC
if len(fpmc_triples) > 0:
    print("\nTraining FPMC...")
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(fpmc_model.parameters(), lr=0.01, weight_decay=0.001)
    FPMC_EPOCHS = 20
    FPMC_BATCH  = 256

    for epoch in range(FPMC_EPOCHS):
        fpmc_model.train()
        perm = torch.randperm(len(fpmc_users))
        users_s = fpmc_users[perm]; prev_s = fpmc_prev[perm]; next_s = fpmc_next[perm]
        epoch_loss = 0; n_batches = 0
        for i in range(0, len(fpmc_users), FPMC_BATCH):
            e = min(i + FPMC_BATCH, len(fpmc_users))
            optimizer.zero_grad()
            scores = fpmc_model(users_s[i:e], prev_s[i:e], next_s[i:e])
            loss = criterion(scores, torch.ones_like(scores))
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss/n_batches:.4f}")

    M = fpmc_model.M.detach().cpu().numpy()
    N = fpmc_model.N.detach().cpu().numpy()
    print(f"✓ FPMC trained: M={M.shape}, N={N.shape}")
else:
    print("No FPMC triples - using random init")
    M = np.random.randn(n_users, K_FPMC) * 0.01
    N = np.random.randn(n_items, K_FPMC) * 0.01

## PHASE 9: Hybrid Fusion & PHASE 10: Tuning

In [ ]:
# Cell 6: Last interaction per user + normalisation helper
print("\n" + "="*60)
print("PHASE 9: HYBRID FUSION")
print("="*60)

if 'timestamp' in train_df.columns:
    last_item = train_df.sort_values('timestamp').groupby('user_idx')['item_idx'].last()
else:
    last_item = train_df.groupby('user_idx')['item_idx'].last()

print(f"Last items extracted for {len(last_item):,} users")

def normalise(arr):
    """Zero-mean unit-std normalisation. Returns zeros if flat."""
    std = arr.std()
    return (arr - arr.mean()) / std if std > 1e-9 else np.zeros_like(arr)

# ── Adaptive weight infrastructure ───────────────────────────────────────────
# omega5(u): trust FPMC more when the user has more interaction history
#            metric: number of (user, item) edges in the training set
# omega6(i): trust KG more when the item is well-connected in the
#            co-interaction graph that node2vec was trained on
#            metric: item node degree in that graph (real edge count)

# --- omega5: user interaction count (directly from train_df) -----------------
user_interaction_counts = train_df.groupby('user_idx').size()
mu_interactions    = user_interaction_counts.mean()
sigma_interactions = user_interaction_counts.std() + 1e-9
print(f"  omega5 | interactions/user: mean={mu_interactions:.1f}  sigma={sigma_interactions:.1f}")

# --- omega6: item degree in the co-interaction graph ------------------------
# Reconstruct the same item-item co-interaction graph that Phase 7 used for
# node2vec: for every user session, connect each pair of consecutive items.
# The degree of an item node = how many distinct neighbours it has, i.e.
# how many other items co-occur with it in user sequences.
# This is the TRUE graph-structural signal, not an embedding norm proxy.
from collections import defaultdict

item_neighbours = defaultdict(set)
train_sorted_tmp = train_df.sort_values(['user_idx', 'timestamp'] if 'timestamp' in train_df.columns else ['user_idx'])

for uid, grp in train_sorted_tmp.groupby('user_idx'):
    items = grp['item_idx'].values.astype(int)
    for i in range(len(items) - 1):
        a, b = items[i], items[i + 1]
        if a != b:
            item_neighbours[a].add(b)
            item_neighbours[b].add(a)

# Build a degree array aligned to item indices 0..n_items-1
item_kg_degree = np.array([len(item_neighbours[i]) for i in range(n_items)], dtype=float)

mu_kg_degree    = item_kg_degree.mean()
sigma_kg_degree = item_kg_degree.std() + 1e-9
n_isolated = int((item_kg_degree == 0).sum())
print(f"  omega6 | KG degree: mean={mu_kg_degree:.1f}  sigma={sigma_kg_degree:.1f}  isolated={n_isolated}")

del item_neighbours, train_sorted_tmp  # free memory

# --- sigmoid + weight functions ----------------------------------------------
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def omega5(user_idx, base_w5=0.3):
    """
    Adaptive FPMC weight for user u.
    w5 = base_w5 * sigmoid( (|R_u| - mu_interactions) / sigma )

    Belief: FPMC learns Markov transitions from sequential history.
    A user with few interactions has noisy transition estimates -> low w5.
    A user with many interactions has reliable transitions -> w5 near base_w5.
    """
    n = user_interaction_counts.get(user_idx, 0)
    z = (n - mu_interactions) / sigma_interactions
    return float(base_w5 * sigmoid(z))

def omega6(prev_item_idx, base_w6=0.3):
    """
    Adaptive KG weight for item i.
    w6 = base_w6 * sigmoid( (deg_G(i) - mu_degree) / sigma )

    Belief: KG similarity (via node2vec) is only reliable when the item
    appears in enough co-occurrence edges to get a well-trained embedding.
    Low-degree items (isolated or rare) -> w6 near 0.
    High-degree items (popular, many co-occurrences) -> w6 near base_w6.
    """
    if prev_item_idx is None or prev_item_idx >= n_items:
        return 0.0
    d = item_kg_degree[prev_item_idx]
    z = (d - mu_kg_degree) / sigma_kg_degree
    return float(base_w6 * sigmoid(z))

print("Adaptive weight functions omega5(u), omega6(i) ready (based on real graph degree)")

# --- last-k items per user (Improvement 4) --------------------------------
K_HISTORY = 3
if 'timestamp' in train_df.columns:
    _sorted = train_df.sort_values(['user_idx', 'timestamp'])
else:
    _sorted = train_df.sort_values(['user_idx'])
last_k_items = _sorted.groupby('user_idx')['item_idx'].apply(
    lambda x: list(x.astype(int).values[-K_HISTORY:])
).to_dict()
print(f"  last-k | K={K_HISTORY}, users: {len(last_k_items):,}")

# --- item popularity (Improvement 5) ----------------------------------------
item_popularity = train_df.groupby('item_idx').size().reindex(
    range(n_items), fill_value=0).values.astype(float)
log_pop      = np.log1p(item_popularity)
log_pop_norm = log_pop / (log_pop.max() + 1e-9)
print(f"  pop    | mean={item_popularity.mean():.1f}, max={item_popularity.max():.0f}")
print("Infrastructure ready: last_k_items, log_pop_norm")


In [ ]:
# Cell 7: Scoring functions (all 6 improvements)
print('='*60)
print('SCORING FUNCTIONS')
print('='*60)

def power_norm(arr, alpha=0.75):
    z = normalise(arr)
    return np.sign(z) * (np.abs(z) ** alpha)

def rrf_score(*score_arrays, k=60):
    fused = np.zeros(n_items)
    for scores in score_arrays:
        if scores.std() < 1e-9: continue
        ranks = np.argsort(np.argsort(-scores))
        fused += 1.0 / (k + ranks)
    return fused

def fpmc_confidence(fpmc_scores, tau=0.5):
    z = normalise(fpmc_scores)
    return (z.max() - z.mean()) >= tau

def fpmc_lastk_scores(user_idx):
    if user_idx >= M.shape[0]: return np.zeros(n_items)
    valid = [j for j in last_k_items.get(user_idx, []) if j < N.shape[0]]
    if not valid: return np.zeros(n_items)
    return np.mean([(M[user_idx] + N[j]) @ N.T for j in valid], axis=0)

def kg_debias(kg_scores, beta=0.3):
    return kg_scores - beta * log_pop_norm

def score_hybrid(user_idx, prev_item_idx,
                 w_fpmc=None, w_kg=None,
                 base_w5=0.3, base_w6=0.3, adaptive=True,
                 alpha_svd=0.75, alpha_fpmc=0.75, alpha_kg=0.75,
                 use_rrf=False, use_lastk=True,
                 use_conf_gate=True, conf_tau=0.5,
                 use_debias=True, debias_beta=0.3):
    svd_scores  = np.zeros(n_items)
    fpmc_scores = np.zeros(n_items)
    kg_scores   = np.zeros(n_items)
    if 0 <= user_idx < P.shape[0]:
        svd_scores = P[user_idx] @ Q.T + bu[user_idx] + bi + r_mean
    if use_lastk:
        fpmc_scores = fpmc_lastk_scores(user_idx)
    elif prev_item_idx is not None and 0 <= user_idx < M.shape[0] and 0 <= prev_item_idx < N.shape[0]:
        fpmc_scores = (M[user_idx] + N[prev_item_idx]) @ N.T
    if prev_item_idx is not None and 0 <= prev_item_idx < n2v_item_emb_norm.shape[0]:
        kg_scores = n2v_item_emb_norm[prev_item_idx] @ n2v_item_emb_norm.T
    if use_debias:
        kg_scores = kg_debias(kg_scores, beta=debias_beta)
    fpmc_trusted = (not use_conf_gate) or fpmc_confidence(fpmc_scores, tau=conf_tau)
    if use_rrf:
        active = [svd_scores]
        if fpmc_trusted: active.append(fpmc_scores)
        active.append(kg_scores)
        return rrf_score(*active)
    svd_n  = power_norm(svd_scores,  alpha=alpha_svd)
    fpmc_n = power_norm(fpmc_scores, alpha=alpha_fpmc) if fpmc_trusted else np.zeros(n_items)
    kg_n   = power_norm(kg_scores,   alpha=alpha_kg)
    if adaptive:
        w5 = omega5(user_idx,      base_w5=base_w5) if fpmc_trusted else 0.0
        w6 = omega6(prev_item_idx, base_w6=base_w6)
    else:
        w5 = (w_fpmc if w_fpmc is not None else base_w5) if fpmc_trusted else 0.0
        w6 =  w_kg   if w_kg   is not None else base_w6
    return svd_n + w5 * fpmc_n + w6 * kg_n

def score_fpmc_only(user_idx, prev_item_idx):
    if prev_item_idx is None or user_idx >= M.shape[0] or prev_item_idx >= N.shape[0]:
        return np.zeros(n_items)
    return (M[user_idx] + N[prev_item_idx]) @ N.T

def score_kg_only(prev_item_idx, w_kg=1.0):
    if prev_item_idx is None or prev_item_idx >= n2v_item_emb_norm.shape[0]:
        return np.zeros(n_items)
    return w_kg * (n2v_item_emb_norm[prev_item_idx] @ n2v_item_emb_norm.T)

print('Scoring suite ready. Helpers: power_norm, rrf_score, fpmc_confidence, fpmc_lastk_scores, kg_debias')


In [ ]:
# Cell 8: NDCG@k metric + unified evaluate function
def ndcg_at_k(rank, k=10):
    return 1.0 / np.log2(rank + 2) if rank < k else 0.0


def evaluate_model(eval_df, scoring_fn, k=10):
    """
    Generic evaluator. `scoring_fn` receives (user_idx, prev_item_idx)
    and must return a score array of length n_items.

    Returns mean NDCG@k.
    """
    ndcgs = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), leave=False):
        uid = int(row['user_idx'])
        iid = int(row['item_idx'])
        if iid >= n_items or iid < 0:
            continue

        prev_iid = last_item.get(uid, None)
        if prev_iid is not None:
            prev_iid = int(prev_iid)
            if prev_iid >= n_items:
                prev_iid = None

        scores = scoring_fn(uid, prev_iid)
        rank   = int(np.sum(scores > scores[iid]))
        ndcgs.append(ndcg_at_k(rank, k))

    return float(np.mean(ndcgs)) if ndcgs else 0.0


print("Evaluation function ready (NDCG@10)")

In [ ]:
# Cell 9: Tune base weights for adaptive hybrid (Phase 10)
print("\n" + "="*60)
print("PHASE 10: ADAPTIVE WEIGHT TUNING (base_w5 x base_w6 grid)")
print("="*60)

# Grid search over base weights; adaptive sigmoid scaling applied inside score_hybrid
w_fpmc_values = [0.1, 0.2, 0.3, 0.4, 0.5]
w_kg_values   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

best_ndcg   = -1
best_w5     = 0.3
best_w6     = 0.3
tuning_log  = []

print(f"  {'base_w5':>8} {'base_w6':>8} {'NDCG@10':>10}")
print("  " + "-"*30)

for bw5 in w_fpmc_values:
    for bw6 in w_kg_values:
        fn   = lambda u, p, _w5=bw5, _w6=bw6: score_hybrid(
                   u, p, adaptive=True, base_w5=_w5, base_w6=_w6)
        ndcg = evaluate_model(val_df, fn, k=10)
        tuning_log.append((bw5, bw6, ndcg))
        marker = " <- best" if ndcg > best_ndcg else ""
        print(f"  {bw5:>8.1f} {bw6:>8.1f} {ndcg:>10.4f}{marker}")
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_w5, best_w6 = bw5, bw6

# Aliases for downstream cells
best_w_kg   = best_w6
best_w_fpmc = best_w5

print(f"\nBest base_w5={best_w5}  base_w6={best_w6}  -> val NDCG@10={best_ndcg:.4f}")
print(f"  (effective per-user/item weights will be <= these via sigmoid scaling)")


In [ ]:
# Cell 9b: Tune power-law alpha per component (Improvement 1)
print('\n' + '='*60)
print('IMPROVEMENT 1: POWER-LAW ALPHA TUNING')
print('='*60)
alpha_values    = [0.5, 0.65, 0.75, 0.85, 1.0]
best_alpha_ndcg = -1
best_alpha_svd  = 0.75
best_alpha_fpmc = 0.75
best_alpha_kg   = 0.75
for label, param in [('alpha_svd', 'svd'), ('alpha_fpmc', 'fpmc'), ('alpha_kg', 'kg')]:
    print(f'\nTuning {label}:')
    for a in alpha_values:
        kw = dict(alpha_svd=best_alpha_svd, alpha_fpmc=best_alpha_fpmc, alpha_kg=best_alpha_kg)
        kw[label] = a
        fn   = lambda u, p, _kw=kw: score_hybrid(u, p, adaptive=True,
                   base_w5=best_w_fpmc, base_w6=best_w_kg,
                   use_rrf=False, use_lastk=False, use_conf_gate=False, use_debias=False, **_kw)
        ndcg = evaluate_model(val_df, fn, k=10)
        mark = ' <- best' if ndcg > best_alpha_ndcg else ''
        print(f'  {label}={a:.2f}  NDCG@10={ndcg:.4f}{mark}')
        if ndcg > best_alpha_ndcg:
            best_alpha_ndcg = ndcg
            if param == 'svd':  best_alpha_svd  = a
            if param == 'fpmc': best_alpha_fpmc = a
            if param == 'kg':   best_alpha_kg   = a
print(f'\nBest: alpha_svd={best_alpha_svd} fpmc={best_alpha_fpmc} kg={best_alpha_kg}  NDCG={best_alpha_ndcg:.4f}')


In [ ]:
# Cell 9c: Cumulative evaluation of Improvements 2-5
print('\n' + '='*60)
print('IMPROVEMENTS 2-5: CUMULATIVE EVAL')
print('='*60)
CONF_TAU_BEST    = 0.5
DEBIAS_BETA_BEST = 0.3
improvement_configs = [
    {'name': 'Baseline (adaptive)',
     'fn': lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=1.0, alpha_fpmc=1.0, alpha_kg=1.0,
         use_rrf=False, use_lastk=False, use_conf_gate=False, use_debias=False)},
    {'name': '+Power norm',
     'fn': lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd, alpha_fpmc=best_alpha_fpmc, alpha_kg=best_alpha_kg,
         use_rrf=False, use_lastk=False, use_conf_gate=False, use_debias=False)},
    {'name': '+Conf gate',
     'fn': lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd, alpha_fpmc=best_alpha_fpmc, alpha_kg=best_alpha_kg,
         use_rrf=False, use_lastk=False, use_conf_gate=True, conf_tau=CONF_TAU_BEST, use_debias=False)},
    {'name': '+KG de-bias',
     'fn': lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd, alpha_fpmc=best_alpha_fpmc, alpha_kg=best_alpha_kg,
         use_rrf=False, use_lastk=False, use_conf_gate=True, conf_tau=CONF_TAU_BEST,
         use_debias=True, debias_beta=DEBIAS_BETA_BEST)},
    {'name': '+Last-k FPMC',
     'fn': lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd, alpha_fpmc=best_alpha_fpmc, alpha_kg=best_alpha_kg,
         use_rrf=False, use_lastk=True, use_conf_gate=True, conf_tau=CONF_TAU_BEST,
         use_debias=True, debias_beta=DEBIAS_BETA_BEST)},
    {'name': 'RRF fusion',
     'fn': lambda u,p: score_hybrid(u,p, use_rrf=True, use_lastk=True,
         use_conf_gate=True, conf_tau=CONF_TAU_BEST, use_debias=True, debias_beta=DEBIAS_BETA_BEST)},
]
improvement_results = {}
imp_base = None
print(f"  {'Config':<22} {'NDCG@10':>10}  {'Delta':>8}")
print('  ' + '-'*44)
for cfg in improvement_configs:
    ndcg = evaluate_model(val_df, cfg['fn'], k=10)
    improvement_results[cfg['name']] = ndcg
    if imp_base is None: imp_base = ndcg
    d = ndcg - imp_base
    print(f"  {cfg['name']:<22} {ndcg:>10.4f}  {d:>+8.4f}")
USE_RRF_FINAL = improvement_results.get('RRF fusion', 0) > improvement_results.get('+Last-k FPMC', 0)
print(f'\nUsing RRF for final model: {USE_RRF_FINAL}')


In [ ]:
# Cell 9d: Improvement 6 - Learned LR stacking
print('\n' + '='*60)
print('IMPROVEMENT 6: LEARNED LR STACKING')
print('='*60)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
_ua_raw  = train_df.groupby('user_idx').size().reindex(range(n_users), fill_value=0).values.astype(float)
_ua_norm = _ua_raw / (_ua_raw.max() + 1e-9)

def _build_feats(df, sample_neg=4):
    rng = np.random.default_rng(42)
    rows, labels = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False, desc='feats'):
        uid = int(row['user_idx']); iid = int(row['item_idx'])
        if iid >= n_items: continue
        prev = last_item.get(uid, None)
        if prev is not None:
            prev = int(prev)
            if prev >= n_items: prev = None
        svd_s  = P[uid] @ Q.T + bu[uid] + bi + r_mean if uid < P.shape[0] else np.zeros(n_items)
        fpm_s  = fpmc_lastk_scores(uid)
        kg_raw = (n2v_item_emb_norm[prev] @ n2v_item_emb_norm.T
                  if prev is not None and prev < n2v_item_emb_norm.shape[0] else np.zeros(n_items))
        kg_s   = kg_debias(kg_raw, beta=DEBIAS_BETA_BEST)
        w5 = omega5(uid); w6 = omega6(prev)
        def _f(i): return [svd_s[i], fpm_s[i], kg_s[i], w5, w6, log_pop_norm[i], _ua_norm[uid]]
        rows.append(_f(iid)); labels.append(1)
        for neg in rng.choice(n_items, size=sample_neg, replace=False):
            rows.append(_f(neg)); labels.append(0)
    return np.array(rows, dtype=np.float32), np.array(labels)

X_tr, y_tr = _build_feats(train_df, 4)
X_va, y_va = _build_feats(val_df,   4)
print(f'Train: {X_tr.shape}  Val: {X_va.shape}')
scaler = StandardScaler().fit(X_tr)
lr_clf = LogisticRegression(C=1.0, max_iter=500, solver='lbfgs')
lr_clf.fit(scaler.transform(X_tr), y_tr)
print(f'Val accuracy: {lr_clf.score(scaler.transform(X_va), y_va):.4f}')
print('LR weights:', dict(zip(['svd','fpmc','kg','w5','w6','log_pop','user_act'], lr_clf.coef_[0].round(3))))

def score_stacked(user_idx, prev_item_idx):
    if user_idx >= P.shape[0]: return np.zeros(n_items)
    svd_s  = P[user_idx] @ Q.T + bu[user_idx] + bi + r_mean
    fpm_s  = fpmc_lastk_scores(user_idx)
    kg_raw = (n2v_item_emb_norm[prev_item_idx] @ n2v_item_emb_norm.T
              if prev_item_idx is not None and prev_item_idx < n2v_item_emb_norm.shape[0]
              else np.zeros(n_items))
    kg_s   = kg_debias(kg_raw, beta=DEBIAS_BETA_BEST)
    w5 = omega5(user_idx); w6 = omega6(prev_item_idx)
    ua = _ua_norm[user_idx] if user_idx < len(_ua_norm) else 0.0
    feats = np.column_stack([svd_s, fpm_s, kg_s,
                             np.full(n_items, w5), np.full(n_items, w6),
                             log_pop_norm, np.full(n_items, ua)]).astype(np.float32)
    return lr_clf.predict_proba(scaler.transform(feats))[:, 1]

stacked_ndcg = evaluate_model(val_df, score_stacked, k=10)
print(f'LR stacking val NDCG@10: {stacked_ndcg:.4f}')
improvement_results['LR stacking'] = stacked_ndcg


## PHASE 11: Extended Ablation Study

Configs tested:

| # | Name | Components |
|---|------|------------|
| 1 | SVD only | Biased MF baseline |
| 2 | **Pure FPMC** | Sequential signal only (no SVD, no KG) |
| 3 | **Pure KG (w=1)** | Cosine KG similarity only (no SVD, no FPMC) |
| 4 | SVD + FPMC | MF + sequential |
| 5 | SVD + KG | MF + structural |
| 6 | SVD + FPMC + KG | Full hybrid |


In [ ]:
# Cell 10: Extended ablation — all 10 configs
print('\n' + '='*60)
print('PHASE 11: EXTENDED ABLATION STUDY')
print('='*60)
ablation_configs = [
    {'name':'SVD only', 'desc':'Matrix factorization only',
     'fn':lambda u,p: score_hybrid(u,p, w_fpmc=0.0, w_kg=0.0,
         alpha_svd=1.0,alpha_fpmc=1.0,alpha_kg=1.0,
         use_rrf=False,use_lastk=False,use_conf_gate=False,use_debias=False)},
    {'name':'Pure FPMC','desc':'Sequential signal only',
     'fn':lambda u,p: score_fpmc_only(u,p)},
    {'name':'Pure KG (w=1)','desc':'KG cosine only',
     'fn':lambda u,p: score_kg_only(p,w_kg=1.0)},
    {'name':'SVD + FPMC','desc':f'Fixed w_fpmc={best_w_fpmc}, no improvements',
     'fn':lambda u,p: score_hybrid(u,p, adaptive=False, w_fpmc=best_w_fpmc, w_kg=0.0,
         alpha_svd=1.0,alpha_fpmc=1.0,alpha_kg=1.0,
         use_rrf=False,use_lastk=False,use_conf_gate=False,use_debias=False)},
    {'name':'SVD + KG','desc':f'Fixed w_kg={best_w_kg}, no improvements',
     'fn':lambda u,p: score_hybrid(u,p, adaptive=False, w_fpmc=0.0, w_kg=best_w_kg,
         alpha_svd=1.0,alpha_fpmc=1.0,alpha_kg=1.0,
         use_rrf=False,use_lastk=False,use_conf_gate=False,use_debias=False)},
    {'name':'Adaptive hybrid','desc':'omega5/omega6 only',
     'fn':lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=1.0,alpha_fpmc=1.0,alpha_kg=1.0,
         use_rrf=False,use_lastk=False,use_conf_gate=False,use_debias=False)},
    {'name':'+Power norm','desc':f'alpha svd={best_alpha_svd} fpmc={best_alpha_fpmc} kg={best_alpha_kg}',
     'fn':lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd,alpha_fpmc=best_alpha_fpmc,alpha_kg=best_alpha_kg,
         use_rrf=False,use_lastk=False,use_conf_gate=False,use_debias=False)},
    {'name':'+Conf+Debias+LastK','desc':'All weighted improvements (1+3+4+5)',
     'fn':lambda u,p: score_hybrid(u,p, adaptive=True, base_w5=best_w_fpmc, base_w6=best_w_kg,
         alpha_svd=best_alpha_svd,alpha_fpmc=best_alpha_fpmc,alpha_kg=best_alpha_kg,
         use_rrf=False,use_lastk=True,
         use_conf_gate=True,conf_tau=CONF_TAU_BEST,use_debias=True,debias_beta=DEBIAS_BETA_BEST)},
    {'name':'RRF fusion','desc':'Improvement 2: rank fusion',
     'fn':lambda u,p: score_hybrid(u,p, use_rrf=True,use_lastk=True,
         use_conf_gate=True,conf_tau=CONF_TAU_BEST,use_debias=True,debias_beta=DEBIAS_BETA_BEST)},
    {'name':'LR stacking','desc':'Improvement 6: logistic regression stacker',
     'fn':score_stacked},
]
print('\nRunning ablation on test set...\n')
ablation_results = {}; svd_baseline = None
for cfg in ablation_configs:
    ndcg = evaluate_model(test_df, cfg['fn'], k=10)
    ablation_results[cfg['name']] = ndcg
    if cfg['name'] == 'SVD only': svd_baseline = ndcg
    delta = ndcg - svd_baseline if svd_baseline is not None else 0.0
    sign  = '+' if delta >= 0 else ''
    print(f"  {cfg['name']:<24} NDCG@10={ndcg:.4f}  ({sign}{delta:.4f} vs SVD)")
    print(f"    {cfg['desc']}")
print(f"\nBest model: {max(ablation_results, key=ablation_results.get)}")


## PHASE 12-13: Full Evaluation & Analysis

In [ ]:
# Cell 11: Cross-split evaluation (best model)
print('\n' + '='*60)
print('PHASE 12-13: FINAL EVALUATION & ANALYSIS')
print('='*60)
best_model_name = max(ablation_results, key=ablation_results.get)
print(f'Best model from ablation: {best_model_name}')
_fn_map = {cfg['name']: cfg['fn'] for cfg in ablation_configs}
best_fn = _fn_map[best_model_name]
splits = {'train': train_df, 'val': val_df, 'test': test_df}
split_results = {}
print('\nEvaluating best model on all splits...')
for split_name, split_df in splits.items():
    ndcg = evaluate_model(split_df, best_fn, k=10)
    split_results[split_name] = ndcg
    print(f'  {split_name.upper()}: NDCG@10 = {ndcg:.4f}')


In [ ]:
# Cell 12: Visualizations
print('\nGenerating visualizations...')
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# -- Plot 1: Full ablation bar chart -----------------------------------------
ax = axes[0]
names  = list(ablation_results.keys())
ndcgs  = list(ablation_results.values())
pal    = ['#c0c0c0','#f4a261','#e76f51','#90b4ce','#90ce90','#4db6ac','#26a69a','#00897b','#ff7043','#7e57c2']
colors = pal[:len(names)]
bars   = ax.bar(range(len(names)), ndcgs, color=colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, ndcgs):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.0003, f'{v:.4f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('NDCG@10', fontsize=11)
ax.set_title('Full ablation study\n(test set)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(ndcgs)*1.14)

# -- Plot 2: Improvement delta waterfall -------------------------------------
ax = axes[1]
svd_base  = ablation_results.get('SVD only', 0)
imp_names = ['Adaptive hybrid', '+Power norm', '+Conf+Debias+LastK', 'RRF fusion', 'LR stacking']
imp_vals  = [ablation_results.get(n, svd_base) for n in imp_names]
deltas    = [v - svd_base for v in imp_vals]
bcolors   = ['#4db6ac' if d >= 0 else '#ef9a9a' for d in deltas]
ax.barh(range(len(imp_names)), deltas, color=bcolors, edgecolor='k', linewidth=0.6)
ax.set_yticks(range(len(imp_names)))
ax.set_yticklabels(imp_names, fontsize=9)
ax.axvline(0, color='k', linewidth=0.8)
for i, d in enumerate(deltas):
    ax.text(d+(0.0002 if d>=0 else -0.0002), i, f'{d:+.4f}',
            va='center', ha='left' if d>=0 else 'right', fontsize=8)
ax.set_xlabel('Delta NDCG@10 vs SVD only', fontsize=10)
ax.set_title('Improvement gains\nvs SVD baseline', fontsize=11)
ax.grid(axis='x', alpha=0.3)

# -- Plot 3: Train/Val/Test --------------------------------------------------
ax = axes[2]
snames = list(split_results.keys()); sndcgs = list(split_results.values())
scols  = ['#aed6f1','#85c1e9','#2980b9']
bars2  = ax.bar(snames, sndcgs, color=scols, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars2, sndcgs):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.0003, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('NDCG@10', fontsize=11)
ax.set_title(f'Best: {best_model_name}\nacross splits', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(sndcgs)*1.12)
plt.tight_layout()
out_fig = f'{OUTPUT_DIR}/final_results_extended.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved {out_fig}')


In [ ]:
# Cell 13: Save final results
print('\nSaving final results...')
results = {
    'ablation_study'    : ablation_results,
    'improvement_study' : improvement_results,
    'split_performance' : split_results,
    'best_model'        : best_model_name,
    'best_w_fpmc'       : float(best_w_fpmc),
    'best_w_kg'         : float(best_w_kg),
    'best_alpha_svd'    : float(best_alpha_svd),
    'best_alpha_fpmc'   : float(best_alpha_fpmc),
    'best_alpha_kg'     : float(best_alpha_kg),
    'tuning_log'        : [(float(a),float(b),float(c)) for a,b,c in tuning_log],
}
with open(f'{OUTPUT_DIR}/final_results_extended.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved final_results_extended.json')
svd_base   = ablation_results['SVD only']
results_df = pd.DataFrame([
    {'Model': name, 'NDCG@10': f'{v:.4f}', 'Delta vs SVD': f'{v-svd_base:+.4f}'}
    for name, v in ablation_results.items()])
results_df.to_csv(f'{OUTPUT_DIR}/results_summary_extended.csv', index=False)
print('Saved results_summary_extended.csv')
print(f"\n{'='*60}\nALL PHASES COMPLETE\n{'='*60}")
print(f'\n{results_df.to_string(index=False)}')
print(f'\nFiles saved to: {OUTPUT_DIR}')
